# DreaMS embeddings and molecular families (first experiment)

**Context.** PRIORITY project (WUR). Meeting with Joana Tirana and Daniel Probst on 19 Oct 2026: embeddings for metabolomics, molecular families, and the DreaMS paper (Bushuiev et al., *Nature Biotechnology*, 2025).
**Author:** Vitória Maia

**Question.** Do DreaMS embeddings of MS/MS spectra put metabolites of the same molecular family close together? And does DreaMS similarity follow chemical structure better than the modified cosine that GNPS uses?

**Steps**
1. Take public GNPS molecular networking jobs on actinobacteria (*Salinispora*, *Streptomyces*) registered in the Paired Omics Data Platform.
2. Clean the spectra with the same quality rules as the DreaMS training data.
3. Compute embeddings with the **pre-trained** DreaMS model (no training in this notebook).
4. **Experiment A:** compare DreaMS neighbours and a DreaMS network with the GNPS molecular families.
5. **Experiment B:** compare DreaMS, modified cosine and a binned spectrum cosine with structural similarity (Tanimoto) on library-annotated spectra.

Each step is a script in `scripts/`. The notebook runs them for every dataset and shows the results. `RESULTS.md` has all the numbers in one page.

## 0. Setup

**One-time steps** (in a terminal, inside `priority-wur/dreams_molecular_families`). DreaMS needs Python 3.11 and fixed versions of numpy and torch, so it gets its own environment. [uv](https://docs.astral.sh/uv/) creates it.
```
git clone https://github.com/pluskal-lab/DreaMS.git .DreaMS
git -C .DreaMS checkout dbec3a0b514a99e5056cfccde4559fda8cfe8129
uv venv .dreams_env --python 3.11
uv pip install --python .dreams_env/Scripts/python.exe -e ./.DreaMS ipykernel nbclient
```
(On Mac/Linux, use `.dreams_env/bin/python`.) The first embedding run downloads the model weights (2.6 GB) from Hugging Face into `.DreaMS/`. Both folders are git-ignored.

**Data.** The GNPS jobs come from `data/raw/gnps/`, which `python -I scripts/download_data.py gnps` fills (run from the repository root). If a job is missing, `prepare_data.py` downloads it.

**Kernel.** Any Jupyter kernel with pandas works: the steps always run with the Python of `.dreams_env`. The simplest choice is the `.dreams_env` kernel itself.

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

RUNS = ["duncan_salinispora", "crusemann_salstrep", "crusemann_streptomyces"]

EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "scripts" / "common.py").exists():
    raise RuntimeError("Open this notebook from the dreams_molecular_families folder.")

# Python used for the steps: this kernel if it has DreaMS, otherwise .dreams_env.
if importlib.util.find_spec("dreams") is not None:
    PY = sys.executable
else:
    PY = str(EXPERIMENT_DIR / ".dreams_env" / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
    if not Path(PY).exists():
        raise RuntimeError("DreaMS environment not found. Do the one-time steps in section 0.")
print("Python used for the steps:", PY)


def step(script, *args, show=False):
    # Run one script and stop with its error message if it fails.
    start = time.time()
    result = subprocess.run(
        [PY, f"scripts/{script}", *args], capture_output=True, text=True,
        encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1"},
    )
    if result.returncode != 0:
        print(result.stdout[-5000:], result.stderr[-5000:], sep="\n")
        raise RuntimeError(f"{script} {' '.join(args)} failed. Read the error above.")
    if show:
        print(result.stdout)
    print(f"done: {script} {' '.join(args)} ({time.time() - start:.0f} s)")


def results(run, name):
    return pd.read_parquet(EXPERIMENT_DIR / "runs" / run / "results" / f"{name}.parquet")


def side_by_side(name, key="metric", value="value"):
    # One summary table per run, merged into one table with a column per run.
    tables = []
    for run in RUNS:
        path = EXPERIMENT_DIR / "runs" / run / "results" / f"{name}.parquet"
        if path.exists():
            tables.append(pd.read_parquet(path).set_index(key)[value].rename(run))
    return pd.concat(tables, axis=1).fillna("")


ready = subprocess.run(
    [PY, "-c", "import torch, dreams; print('DreaMS is ready. GPU available:', torch.cuda.is_available())"],
    capture_output=True, text=True, check=True,
)
print(ready.stdout.strip())
pd.set_option("display.max_colwidth", 80)

## 1. Data

| Run | Dataset | MassIVE (PoDP) | GNPS job |
|---|---|---|---|
| `duncan_salinispora` | *Salinispora*, Duncan et al. 2015 | MSV000079284 | `c22f44b1…` (METABOLOMICS-SNETS). Example job of the NPLinker docs, created by Justin van der Hooft (WUR) |
| `crusemann_salstrep` | *Salinispora* + *Streptomyces*, Crüsemann et al. 2017 | MSV000078836 | `9360fa51…` (METABOLOMICS-SNETS) |
| `crusemann_streptomyces` | *Streptomyces*, Crüsemann et al. 2017 | MSV000078839 | `840eadb0…` (METABOLOMICS-SNETS-V2) |

All three are the GNPS jobs that the Paired Omics Data Platform links to genomes, so the spectra can later be connected to BGCs with NPLinker. The two Crüsemann jobs overlap: the SalStrep job also re-analyses MSV000078839.

**Molecular families.** GNPS connects two spectra when their modified cosine is high enough. Each connected group is a molecular family (`componentindex`). Spectra without connections are singletons.

**Cleaning.** We keep only spectra that follow DreaMS "format A", the format of the data DreaMS was pre-trained on: charge +1, precursor m/z ≤ 1000, at least 3 peaks above 10% intensity, and max/min intensity ≥ 20. Peaks above the precursor m/z are removed, and at most 128 peaks are kept. The thresholds are at the top of `scripts/common.py`.

To add another dataset, add it to `RUNS` in `scripts/common.py` (any GNPS classic molecular networking job) and to `RUNS` in the setup cell.

In [ ]:
for run in RUNS:
    step("prepare_data.py", "--run", run)

info = pd.DataFrame({run: pd.read_json(EXPERIMENT_DIR / "runs" / run / "results" / "run_info.json", typ="series") for run in RUNS})
display(info.drop(["run", "clean MGF md5"]))
display(side_by_side("00_filter_report", key="step", value="spectra"))

## 2. DreaMS embeddings

We use the pre-trained `DreaMS_embedding` model. It gives one vector of 1024 numbers per spectrum. Spectra that DreaMS sees as similar get vectors that point in a similar direction (high cosine similarity).

On a laptop CPU this takes about 4 spectra per second. The embeddings are saved in `runs/<run>/data/dreams_embeddings.npy` and are only computed again if the cleaned spectra change.

In [ ]:
for run in RUNS:
    step("embed_dreams.py", "--run", run)

display(pd.DataFrame({run: pd.read_json(EXPERIMENT_DIR / "runs" / run / "data" / "dreams_embeddings.json", typ="series") for run in RUNS}))

## 3. Experiment A: molecular families

- **A1. Neighbours.** For each spectrum in a GNPS family with at least 3 members, we look at its 5 nearest neighbours in DreaMS space. *Family recovery@5* = family partners found, divided by the best possible number. Three baselines: the cosine of binned spectra (1 Da bins, no learning), neighbours chosen only by precursor m/z, and random neighbours.
- **A2. GNPS edges.** Pairs connected in the GNPS network should have a higher DreaMS similarity than random pairs.
- **A3. DreaMS network.** Each spectrum is linked to its 3 nearest neighbours if the DreaMS similarity is at least 0.7 (as in the DreaMS networking tutorial). We compare the resulting families with the GNPS families (AMI/ARI). DreaMS links that involve GNPS singletons are saved in `results/exp_a_new_links.parquet`. They are hypotheses to check by hand, not results.
- **A4. Network thresholds.** The same DreaMS network with thresholds from 0.6 to 0.95, to see how the size of the largest family and the agreement with GNPS change.

⚠️ GNPS families come from modified cosine, so this experiment measures **agreement with GNPS**, not truth. Experiment B uses chemical structures instead.

In [ ]:
for run in RUNS:
    step("exp_a_families.py", "--run", run)

display(side_by_side("exp_a_summary"))
for run in RUNS:
    display(Markdown(f"**A4. DreaMS network thresholds: `{run}`**"))
    display(results(run, "exp_a_network_sweep"))

In [ ]:
for run in RUNS:
    display(Markdown(f"### `{run}`"))
    display(Image(EXPERIMENT_DIR / "runs" / run / "results" / "exp_a_umap.png", width=560))
    display(Image(EXPERIMENT_DIR / "runs" / run / "results" / "exp_a_similarity_hist.png", width=460))

In [ ]:
# DreaMS links that GNPS did not make (first rows). Look at the delta m/z
# (for example +14 = CH2, +16 = O) and at the library names.
for run in RUNS:
    display(Markdown(f"**New links: `{run}`**"))
    display(results(run, "exp_a_new_links").head(10))

## 4. Experiment B: chemical structures

We use only spectra with a GNPS library hit that has a structure (SMILES). For every pair of spectra, we compare the structural similarity of the two molecules (Tanimoto on Morgan fingerprints) with three spectral scores: DreaMS similarity, modified cosine (same fragment tolerance as the GNPS job) and binned spectrum cosine.

- **B1.** Spearman correlation over all pairs.
- **B2.** For each spectrum, take its most similar other spectrum and check the Tanimoto of the two molecules. *Analog search* skips spectra of the same molecule, which is closer to the idea of molecular families.

In [ ]:
for run in RUNS:
    step("exp_b_structures.py", "--run", run)

display(side_by_side("exp_b_summary"))
for run in RUNS:
    figure = EXPERIMENT_DIR / "runs" / run / "results" / "exp_b_structure.png"
    if figure.exists():
        display(Image(figure, width=900))

## 5. All results in one page

`scripts/report.py` writes `RESULTS.md` with the tables and figures of every run.

In [ ]:
step("report.py", "--runs", *RUNS)

## 6. How to read the results

Write your own interpretation here before the meeting. Some guiding questions:
- In A1, is DreaMS clearly above the binned cosine, the precursor m/z baseline and the random level? Is it the same in the three datasets?
- In A3/A4, which threshold gives families that look like the GNPS families? What happens to the largest family?
- In B1 and B2, is DreaMS better than modified cosine, or about the same? How many annotated spectra does each dataset have?
- Do some links in `exp_a_new_links.parquet` make chemical sense? Look at the Δm/z (for example +14 = CH2, +16 = O) and at the library names.

**My interpretation:** *(write here)*

## 7. Limitations

1. **GNPS families are not ground truth.** They come from modified cosine, so Experiment A measures agreement with GNPS.
2. **Library hits are putative.** They are spectral matches, not confirmed structures. Experiment B has few annotated spectra in the Crüsemann jobs.
3. **Data quality and resolution.** Many spectra are removed by the quality rules (see the filter reports). The GNPS jobs used fragment tolerances of 0.3 Da (Duncan) and 0.5 Da (Crüsemann), which suggests low-resolution data or wide settings, while DreaMS was pre-trained on data with high mass accuracy. Check the instruments on MassIVE. If the data are low resolution, DreaMS is used outside its training domain.
4. **Possible overlap with DreaMS training data.** These datasets are public on MassIVE, so they may be part of GeMS (DreaMS pre-training, without labels). The embedding model was also fine-tuned on spectra with known structures; if the same compounds were used, Experiment B can look better than it really is.
5. **The runs are not independent.** The SalStrep job also re-analyses the *Streptomyces* data of MSV000078839.
6. **Small numbers.** This is a first look, not a conclusion.

## 8. Open questions and next steps

**Questions for Joana and Daniel**
1. Should I keep the pre-trained DreaMS model, fine-tune it, or train a new model?
2. Which dataset should I use? Is there a high-resolution actinobacteria dataset in PRIORITY or in PoDP?
3. What is the end goal for the metabolomics layer: molecular families, linking spectra to BGCs (NPLinker), or features for a multi-omics model?

**Next steps (proposal)**
- Run this notebook on a high-resolution actinobacteria dataset (add it to `RUNS`).
- Use DreaMS families instead of GNPS families in NPLinker-style linking to gene cluster families.